In [ ]:
import pandas as pd
from google.colab import files

# Step 1: Upload the 4 Excel files from your local system
uploaded = files.upload()

# Step 2: Read and merge the Excel files
filenames = ["Final Dataset1.xlsx", "Final Dataset2.xlsx", "Final Dataset3.xlsx", "Final Dataset4.xlsx"]

# Read and concatenate
merged_df = pd.concat([pd.read_excel(file) for file in filenames], ignore_index=True)

# Step 3: Save the merged file
merged_df.to_excel("Merged_Final_Dataset.xlsx", index=False)

# Step 4: Download the merged file
files.download("Merged_Final_Dataset.xlsx")


In [ ]:
# Step 5: Define approved newspaper names
approved_newspapers = [
    "The Hindu", "Hindustan Times", "Indian Express", "The Telegraph", "Deccan Chronicle",
    "The New Indian Express", "Mint", "Business Standard", "Financial Express",
    "DNA (Daily News and Analysis)", "The Tribune", "The Statesman", "The Asian Age",
    "The Pioneer", "The Free Press Journal", "The Economic Times", "The Afternoon Despatch & Courier",
    "The Sentinel", "The Navhind Times", "Goa Chronicle", "The Assam Tribune",
    "The Arunachal Times", "The Shillong Times", "The Imphal Free Press", "The Sikkim Express",
    "The Hans India", "The Orissa Post", "The Daily Post (Chandigarh)", "The Hitavada",
    "The Meghalaya Guardian", "The Morung Express", "The Sangai Express", "The Arunachal Front"
]

# Step 6: Install and import fuzzy matching tools
!pip install fuzzywuzzy python-Levenshtein openpyxl --quiet

from fuzzywuzzy import process, fuzz

# Step 7: Define the mapping function
def force_map(name):
    if pd.isna(name) or not isinstance(name, str) or name.strip() == "":
        return "other_news"
    name = name.strip()
    match, score = process.extractOne(name, approved_newspapers, scorer=fuzz.token_set_ratio)
    return match

# Step 8: Apply the mapping
merged_df['newspaper_name'] = merged_df['newspaper_name'].apply(force_map)

# Step 9: Save the mapped dataset
final_output = "Mapped_Merged_Final_Dataset.xlsx"
merged_df.to_excel(final_output, index=False)

# Step 10: Download the mapped file
files.download(final_output)


In [ ]:
# Step 11: Standardize the date format
merged_df['publication_date'] = pd.to_datetime(merged_df['publication_date'], errors='coerce').dt.strftime('%Y-%m-%d')

# Step 12: Drop rows with any missing value in any column
merged_df.dropna(inplace=True)

# Step 13: Save final cleaned dataset
final_cleaned_output = "Cleaned_Mapped_Merged_Final_Dataset.xlsx"
merged_df.to_excel(final_cleaned_output, index=False)

# Step 14: Download final file
files.download(final_cleaned_output)


In [ ]:
# Step 16: Detect headlines with only one word
single_word_headlines = merged_df[merged_df['headline'].apply(lambda x: isinstance(x, str) and len(x.split()) == 1)]

# Step 17: Save log of single-word headlines
single_word_headlines.to_excel("Single_Word_Headline_Log.xlsx", index=False)
print(f"Number of single-word headlines removed: {len(single_word_headlines)}")

# Step 18: Remove them from main DataFrame
merged_df = merged_df[~merged_df.index.isin(single_word_headlines.index)]


In [ ]:
files.download("Single_Word_Headline_Log.xlsx")


In [ ]:
final_cleaned_filename = "Final_Cleaned_Dataset.xlsx"
merged_df.to_excel(final_cleaned_filename, index=False)

# Step 21: Download the cleaned file
files.download(final_cleaned_filename)

In [ ]:
!pip install transformers --quiet


from transformers import pipeline
from tqdm import tqdm

# Define your approved categories
approved_categories = [
    "International News", "National News", "Local News", "Politics", "Business and Finance",
    "Science and Technology", "Health and Wellness", "Entertainment", "Sports", "Lifestyle and Features",
    "Opinion and Editorial", "Environment", "Education", "Crime and Justice", "Human Interest", "Obituaries",
    "Weather", "Religion and Spirituality", "Technology and Gadgets", "Automotive", "Other…"
]

# Load the zero-shot classification model
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

# Combine 'headline' and 'article_content' safely into one text column
df['text_for_classification'] = (
    df['headline'].fillna('').astype(str) + ". " +
    df['article_content'].fillna('').astype(str)
)

predicted_categories = []

for text in tqdm(df['text_for_classification'], desc="Classifying News Categories"):
    try:
        result = classifier(text, approved_categories, multi_label=False)
        predicted_categories.append(result['labels'][0])
    except:
        predicted_categories.append("Other…")  # fallback in case of error

# Add predicted column
df['predicted_news_category'] = predicted_categories

# Replace original if you want
df['news_category'] = df['predicted_news_category']
df.drop(columns=['predicted_news_category', 'text_for_classification'], inplace=True)

df.to_excel("Final_Category_Mapped_Dataset.xlsx", index=False)
files.download("Final_Category_Mapped_Dataset.xlsx")


In [ ]:
from fuzzywuzzy import process
import numpy as np

# Step 22: Define approved news categories
approved_categories = [
    "International News", "National News", "Local News", "Politics", "Business and Finance",
    "Science and Technology", "Health and Wellness", "Entertainment", "Sports", "Lifestyle and Features",
    "Opinion and Editorial", "Environment", "Education", "Crime and Justice", "Human Interest", "Obituaries",
    "Weather", "Religion and Spirituality", "Technology and Gadgets", "Automotive", "Other…"
]

# Step 23: Function to fuzzy match categories
def map_category(cat):
    if pd.isnull(cat) or not isinstance(cat, str):
        return "Other…"
    best_match, score = process.extractOne(cat.strip(), approved_categories)
    return best_match if score >= 70 else "Other…"

# Step 24: Create a log of original vs mapped categories
merged_df['original_news_category'] = merged_df['news_category']
merged_df['news_category'] = merged_df['news_category'].apply(map_category)

# Step 25: Save mapping log for review
category_mapping_log = merged_df[['original_news_category', 'news_category']].drop_duplicates()
category_mapping_log.to_excel("News_Category_Mapping_Log.xlsx", index=False)

# Step 26: Download the mapping log
files.download("News_Category_Mapping_Log.xlsx")
